In [1]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np 
import torch

train_data = ProteinDataset(path_to_data="../../../data/train.csv")
val_data = ProteinDataset(path_to_data="../../../data/val.csv")
k = 2
def make_windows(k, dataset):
     
    all_windows = []
    all_targets = []

    for row_idx in range(len(dataset)):
        
        row = dataset.tokens[row_idx].tolist()
        target = dataset.tokens_shifted[row_idx].tolist()
        
        padded = [0] * (k-1) + row
        
        for i in range(0, len(row)):
            if target[i] == -1: 
                break
                
            window = padded[i:i+k]
            all_windows.append(window)
            all_targets.append(target[i])
    return torch.tensor(all_windows, dtype=torch.long), torch.tensor(all_targets, dtype=torch.long)

X_train, y_train = make_windows(k, train_data)
X_val, y_val = make_windows(k, val_data)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)





torch.Size([1537909, 2]) torch.Size([1537909])
torch.Size([191204, 2]) torch.Size([191204])


In [12]:
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, vocab_size, k, embed_dim=16, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(k * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        emb = self.embedding(x).flatten(start_dim=1)
        return self.net(emb)


In [13]:
model = MLP(train_data.vocab_size, k)
xb, yb = next(iter(train_loader))
logits = model(xb)
print(logits.shape)                 # torch.Size([32, 23])
print(F.cross_entropy(logits, yb))  # about 3.1 before training

torch.Size([32, 23])
tensor(3.2598, grad_fn=<NllLossBackward0>)


In [15]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    train_total = 0
    for xb, yb in train_loader:
        loss = F.cross_entropy(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_total += loss.item() * len(xb)

    model.eval()
    val_total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            val_total += F.cross_entropy(model(xb), yb, reduction="sum").item()

    train_loss = train_total / len(X_train)
    val_loss = val_total / len(X_val)
    print(f"epoch {epoch}: train loss {train_loss:.3f}, val loss {val_loss:.3f}")


epoch 0: train loss 2.530, val loss 2.544
epoch 1: train loss 2.527, val loss 2.540
epoch 2: train loss 2.525, val loss 2.540
epoch 3: train loss 2.523, val loss 2.542
epoch 4: train loss 2.522, val loss 2.537
